# 05. 분류 모델 튜닝 (Optuna + SMOTE)

`04`의 기본 분류를 바탕으로, **폐업률 상위 25%를 '위험'(1)** 으로 라벨링하고
SMOTE로 불균형을 보정한 뒤 **optuna로 하이퍼파라미터를 튜닝**한 최종 분류 모델을 정리한다.
각 섹션은 독립 실행되며 `../data/eda/merged_data.csv`를 로드한다.

| 모델 | 튜닝 | 불균형 | 비고 |
|------|------|--------|------|
| RandomForest | optuna | SMOTE | StratifiedKFold 검증 |
| CatBoost | optuna | SMOTE | 최종 모델 저장(pkl) |
| XGBoost | optuna | SMOTE | StratifiedKFold 검증 |
| LightGBM | optuna | 파이프라인 내 SMOTE | 임계값(F1) 튜닝 |

> optuna 탐색 횟수는 재현/실행 시간을 고려해 축소했다(원본 50 → 15). 구조는 동일하다.

## 0. 공통 설정

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import optuna

import matplotlib.font_manager as fm
import matplotlib
font_path = 'C:\\Windows\\Fonts\\gulim.ttc'
font = fm.FontProperties(fname=font_path).get_name()
matplotlib.rc('font', family=font)
plt.rcParams['axes.unicode_minus'] = False

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_PATH = '../data/eda/merged_data.csv'
N_TRIALS = 15  # 원본 50에서 축소

## 1. RandomForest (optuna)

`04`의 RF objective는 RandomForest에 없는 `learning_rate`/`colsample_bytree`를 탐색하고 모델에 반영하지 않는 버그가 있었다. RF에 유효한 파라미터로 바로잡았다.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv(DATA_PATH)
df = df.drop(['폐업_점포_수', '폐업_영업_개월_평균', '서울시_폐업_영업_개월_평균', '기준_년분기_코드'], axis=1)

# 레이블 인코딩
for col in ['자치구_코드_명', '서비스_업종_코드_명', '상권_변화_지표']:
    df[col] = LabelEncoder().fit_transform(df[col])

# 상위 25% → 위험(1)
df['폐업률_등급'] = pd.qcut(df['폐업_률'], q=[0, 0.75, 1.0], labels=[0, 1])
X = df.drop(['폐업_률', '폐업률_등급'], axis=1)
y_class = df['폐업률_등급']

# SMOTE 후 분할
X_res, y_res = SMOTE(random_state=42).fit_resample(X, y_class)
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# optuna 목적함수 (RF 유효 파라미터로 수정, 실제 모델에 주입)
def rf_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=100),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 4),
        'random_state': 42,
        'n_jobs': -1,
    }
    clf = RandomForestClassifier(**params)
    return cross_val_score(clf, X_train, y_train, scoring='accuracy', cv=3, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(rf_objective, n_trials=N_TRIALS)
print('Best CV accuracy:', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
# 최적 파라미터로 최종 학습/평가
best_rf = RandomForestClassifier(**study.best_params, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)
y_pred = best_rf.predict(X_test)

print(f'ACC : {accuracy_score(y_test, y_pred):.4f}')
print(f'PREC: {precision_score(y_test, y_pred):.4f}')
print(f'REC : {recall_score(y_test, y_pred):.4f}')
print(f'F1  : {f1_score(y_test, y_pred):.4f}')
print(f'AUC : {roc_auc_score(y_test, best_rf.predict_proba(X_test)[:, 1]):.4f}')
print(classification_report(y_test, y_pred, target_names=['안정', '위험']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['안정', '위험'], yticklabels=['안정', '위험'])
plt.xlabel('예측값'); plt.ylabel('실제값'); plt.title('RandomForest Confusion Matrix')
plt.show()

## 2. CatBoost (optuna) — 최종 모델

optuna 튜닝 후 최종 모델을 학습하고 인코더와 함께 `pkl`로 저장한다 (서비스 배포용).

In [ ]:
from catboost import CatBoostClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, classification_report,
                             confusion_matrix, roc_curve, precision_recall_curve)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv(DATA_PATH)
df = df.drop(['폐업_점포_수', '폐업_영업_개월_평균', '서울시_폐업_영업_개월_평균', '기준_년분기_코드'], axis=1)

# 레이블 인코딩
for col in ['자치구_코드_명', '서비스_업종_코드_명', '상권_변화_지표']:
    df[col] = LabelEncoder().fit_transform(df[col])

# 상위 25% → 위험(1)
df['폐업률_등급'] = pd.qcut(df['폐업_률'], q=[0, 0.75, 1.0], labels=[0, 1])
X = df.drop(['폐업_률', '폐업률_등급'], axis=1)
y_class = df['폐업률_등급']

# SMOTE 후 분할
X_res, y_res = SMOTE(random_state=42).fit_resample(X, y_class)
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# optuna 목적함수
def cat_objective(trial):
    params = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, step=0.01),
        'iterations': trial.suggest_int('iterations', 300, 800, step=100),
        'l2_leaf_reg': trial.suggest_int('l2_leaf_reg', 1, 10),
        'random_strength': trial.suggest_int('random_strength', 1, 10),
        'border_count': trial.suggest_categorical('border_count', [32, 64, 128]),
        'verbose': 0, 'random_state': 42,
    }
    model = CatBoostClassifier(**params)
    return cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy', n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(cat_objective, n_trials=N_TRIALS)
print('Best CV accuracy:', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
# 최종 모델 학습/평가
final_model = CatBoostClassifier(**study.best_params, verbose=0, random_state=42)
final_model.fit(X_train, y_train)

y_pred = final_model.predict(X_test)
y_proba = final_model.predict_proba(X_test)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'Precision: {precision_score(y_test, y_pred):.4f}')
print(f'Recall   : {recall_score(y_test, y_pred):.4f}')
print(f'F1       : {f1_score(y_test, y_pred):.4f}')
print(f'AUC      : {roc_auc_score(y_test, y_proba):.4f}')
print(classification_report(y_test, y_pred, target_names=['안정', '위험']))

In [ ]:
# 혼동행렬 / ROC / PR
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low', 'High'], yticklabels=['Low', 'High'], ax=axes[0])
axes[0].set_xlabel('예측값'); axes[0].set_ylabel('실제값'); axes[0].set_title('Confusion Matrix')

fpr, tpr, _ = roc_curve(y_test, y_proba)
axes[1].plot(fpr, tpr, label=f'AUC = {roc_auc_score(y_test, y_proba):.2f}')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('ROC Curve'); axes[1].legend()

prec, rec, _ = precision_recall_curve(y_test, y_proba)
axes[2].plot(rec, prec, marker='.')
axes[2].set_xlabel('Recall'); axes[2].set_ylabel('Precision'); axes[2].set_title('PR Curve')
plt.tight_layout(); plt.show()

In [ ]:
# 특성 중요도 (Top 20)
importances_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': final_model.get_feature_importance()
}).sort_values(by='Importance', ascending=False)
plt.figure(figsize=(8, 6))
sns.barplot(data=importances_df.head(20), x='Importance', y='Feature', palette='viridis')
plt.title('CatBoost 특성 중요도 (Top 20)')
plt.tight_layout(); plt.show()

### 최종 모델 저장

모델 + 인코더 + 데이터를 함께 저장해 Streamlit 서비스에서 재사용한다.

In [ ]:
import joblib
from sklearn.preprocessing import LabelEncoder

# 인코더 재생성 (원본 범주값 기준)
df_original = pd.read_csv(DATA_PATH)
district_encoder = LabelEncoder().fit(df_original['자치구_코드_명'])
industry_encoder = LabelEncoder().fit(df_original['서비스_업종_코드_명'])
sanggwon_encoder = LabelEncoder().fit(df_original['상권_변화_지표'])

joblib.dump({
    'model': final_model,
    'district_encoder': district_encoder,
    'industry_encoder': industry_encoder,
    'sanggwon_encoder': sanggwon_encoder,
    'X_train': X_train, 'y_train': y_train,
    'X_test': X_test, 'y_test': y_test,
    'feature_names': X_train.columns.tolist()
}, '../model/catboost/model_and_data_encoded.pkl')
print('최종 모델 저장 완료, feature 수:', len(X_train.columns))

## 3. XGBoost (optuna)

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, classification_report, confusion_matrix,
                             roc_curve, auc)

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

df = pd.read_csv(DATA_PATH)
df = df.drop(['폐업_점포_수', '폐업_영업_개월_평균', '서울시_폐업_영업_개월_평균', '기준_년분기_코드'], axis=1)

# 레이블 인코딩
for col in ['자치구_코드_명', '서비스_업종_코드_명', '상권_변화_지표']:
    df[col] = LabelEncoder().fit_transform(df[col])

# 상위 25% → 위험(1)
df['폐업률_등급'] = pd.qcut(df['폐업_률'], q=[0, 0.75, 1.0], labels=[0, 1])
X = df.drop(['폐업_률', '폐업률_등급'], axis=1)
y_class = df['폐업률_등급']

# SMOTE 후 분할
X_res, y_res = SMOTE(random_state=42).fit_resample(X, y_class)
X_train, X_test, y_train, y_test = train_test_split(
    X_res, y_res, test_size=0.2, random_state=42, stratify=y_res)
print('Train:', X_train.shape, 'Test:', X_test.shape)

In [ ]:
# optuna 목적함수
def xgb_objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500, step=100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'random_state': 42,
    }
    clf = XGBClassifier(**params)
    return cross_val_score(clf, X_train, y_train, scoring='accuracy', cv=3, n_jobs=-1).mean()

study = optuna.create_study(direction='maximize')
study.optimize(xgb_objective, n_trials=N_TRIALS)
print('Best CV accuracy:', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
# StratifiedKFold 검증 + 최종 평가
best_xgb = XGBClassifier(**study.best_params, random_state=42)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_acc = []
Xtr_np, ytr_np = X_train.to_numpy(), y_train.to_numpy()
for tr_idx, val_idx in skf.split(Xtr_np, ytr_np):
    best_xgb.fit(Xtr_np[tr_idx], ytr_np[tr_idx])
    cv_acc.append(accuracy_score(ytr_np[val_idx], best_xgb.predict(Xtr_np[val_idx])))
print('CV 평균 정확도:', round(np.mean(cv_acc), 4))

best_xgb.fit(X_train, y_train)
y_pred = best_xgb.predict(X_test)
print(classification_report(y_test, y_pred, target_names=['안정', '위험']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d',
            xticklabels=['안정(0)', '위험(1)'], yticklabels=['안정(0)', '위험(1)'], ax=axes[0])
axes[0].set_xlabel('예측값'); axes[0].set_ylabel('실제값'); axes[0].set_title('XGBoost Confusion Matrix')

y_prob = best_xgb.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color='blue', label=f'AUC={auc(fpr, tpr):.2f}')
axes[1].plot([0, 1], [0, 1], color='gray', linestyle='--')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR'); axes[1].set_title('XGBoost ROC'); axes[1].legend()
plt.tight_layout(); plt.show()

## 4. LightGBM (optuna + 파이프라인)

전처리 → SMOTE → LGBM을 하나의 파이프라인으로 묶어 **SMOTE가 학습 폴드에서만** 적용되게 하고, PR-AUC 기준 optuna 튜닝 후 검증셋에서 F1이 최대가 되는 임계값을 찾아 테스트에 적용한다.

In [ ]:
import lightgbm as lgb
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             precision_recall_curve)

RANDOM_STATE = 42
df = pd.read_csv(DATA_PATH)
df = df.dropna(subset=['폐업_률']).copy()

# 상위 25% 라벨
cut_75 = df['폐업_률'].quantile(0.75)
df['폐업률_등급'] = (df['폐업_률'] > cut_75).astype(int)
df = df.drop(columns=['폐업_점포_수', '폐업_영업_개월_평균', '서울시_폐업_영업_개월_평균', '기준_년분기_코드'],
             errors='ignore')
print('임계값(75%):', round(cut_75, 4), '| 라벨 분포:', df['폐업률_등급'].value_counts().to_dict())

In [ ]:
X = df.drop(columns=['폐업_률', '폐업률_등급'], errors='ignore')
y = df['폐업률_등급'].astype(int)

cat_cols = [c for c in X.columns if X[c].dtype.name in ['object', 'category']]
num_cols = [c for c in X.columns if c not in cat_cols]

preprocess = ColumnTransformer([
    ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat_cols),
    ('num', 'passthrough', num_cols),
], remainder='drop')

# train 0.7 / val 0.1 / test 0.2
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
X_train, X_val, y_train, y_val = train_test_split(X_trval, y_trval, test_size=0.125, stratify=y_trval, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Val:', X_val.shape, 'Test:', X_test.shape)

In [ ]:
pipeline = ImbPipeline([
    ('prep', preprocess),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('clf', lgb.LGBMClassifier(objective='binary', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1)),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def lgbm_objective(trial):
    params = {
        'clf__n_estimators': trial.suggest_int('clf__n_estimators', 300, 1500),
        'clf__learning_rate': trial.suggest_float('clf__learning_rate', 0.01, 0.1, log=True),
        'clf__num_leaves': trial.suggest_int('clf__num_leaves', 31, 150),
        'clf__max_depth': trial.suggest_int('clf__max_depth', -1, 12),
        'clf__min_child_samples': trial.suggest_int('clf__min_child_samples', 20, 200),
        'clf__subsample': trial.suggest_float('clf__subsample', 0.6, 1.0),
        'clf__colsample_bytree': trial.suggest_float('clf__colsample_bytree', 0.6, 1.0),
        'clf__reg_lambda': trial.suggest_float('clf__reg_lambda', 0.0, 2.0),
    }
    pipe = pipeline.set_params(**params)
    scores = cross_val_score(pipe, X_train, y_train, scoring='average_precision', cv=cv, n_jobs=-1)
    return float(np.mean(scores))

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction='maximize', sampler=sampler)
study.optimize(lgbm_objective, n_trials=N_TRIALS)
print('Best CV PR-AUC:', round(study.best_value, 4))
print('Best params:', study.best_params)

In [ ]:
# 최적 파이프라인 학습
best_pipe = pipeline.set_params(**study.best_params)
best_pipe.fit(X_train, y_train)

# 검증셋에서 F1 최대 임계값 찾기
val_proba = best_pipe.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, val_proba)
f1s = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-12)
best_thr = float(thresholds[int(np.nanargmax(f1s))])
print(f'Best threshold (VAL F1 max): {best_thr:.4f}')

In [ ]:
def eval_all(y_true, proba, thr):
    pred = (proba >= thr).astype(int)
    return {
        'ACC': round(accuracy_score(y_true, pred), 4),
        'PREC': round(precision_score(y_true, pred, zero_division=0), 4),
        'RECALL': round(recall_score(y_true, pred, zero_division=0), 4),
        'F1': round(f1_score(y_true, pred, zero_division=0), 4),
        'ROC_AUC': round(roc_auc_score(y_true, proba), 4),
        'PR_AUC': round(average_precision_score(y_true, proba), 4),
    }

test_proba = best_pipe.predict_proba(X_test)[:, 1]
print('=== VAL ===', eval_all(y_val, val_proba, best_thr))
print('=== TEST ==', eval_all(y_test, test_proba, best_thr))

cm = confusion_matrix(y_test, (test_proba >= best_thr).astype(int))
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['안정', '위험'], yticklabels=['안정', '위험'])
plt.xlabel('예측값'); plt.ylabel('실제값'); plt.title('LightGBM Confusion Matrix')
plt.show()

## 정리

optuna + SMOTE 튜닝으로 네 모델의 분류 성능을 끌어올렸다. 그중 **CatBoost를 최종 모델로 채택**해
인코더와 함께 저장했고, 이후 Streamlit 서비스에서 이 pkl을 로드해 사용한다.